# Jump Filter · Interactive bond trade review

This notebook uses generated trades to demonstrate local spread outliers,
genuine level changes and sparse trading. No private bond records are included.
From this repository, install `python -m pip install -e ".[notebook,speed]"` and launch
`python -m jupyterlab jump_filter_dashboard.ipynb` or `python -m notebook`.
Select the Python kernel from that environment, then choose **Run All Cells**.
Existing Jupyter installations can install `%pip install -e ".[dashboard,speed]"`
in their kernel, restart it, and rerun this notebook. The notebook uses native
ipywidgets and Plotly FigureWidget with locally bundled JavaScript; no CDN or
separate Streamlit server is required. Controls need a live Python kernel.

Pick a CUSIP and method, edit the sliders or exact inputs, then **Apply filter**.
The statistical dashboard and CSV/settings exports use the last successful Apply.
Selected bond is the default review population. Choose All bonds · batch and
Apply explicitly for complete-source annotations and portfolio statistics.
CUSIP revisits reuse a bounded cache; changing methods stays pending until Apply.
**Method & mathematics** immediately follows your selected method, even before Apply.
It renders formulas, numerical examples, parameter effects and failure/coverage rules.
Historical fit screening may use both earlier and later trades; Causal EWMA is an
optional online comparator. Methods include Hampel, IQR fences, robust local linear,
reversal, multiscale Hampel, two-sided local piecewise trend, offline Huber-TV
level and conservative consensus.
The trading-calendar controls remove configured closures from time distances;
local support, volatility, reference density and active gaps remain auditable.
Optional fitting rules add quantity-aware handling of uncertain trades and a
maximum distance from a supported trend. They are off initially; the generated
`Quantity` column lets you inspect the controls without a private trade file.

In [ ]:
# SETUP LOGIC: Imports create no filtering result and read no private dataset.
from jump_filter import FilterConfig, make_demo, show_filter, select_fit_data

In [ ]:
# SETUP LOGIC: The reproducible generator is a public synthetic demonstration.
data = make_demo(seed=42)

# CONFIGURATION LOGIC: Spread values are already basis points; the unit label applies no scaling.
settings = FilterConfig(method="consensus", window=31, horizon="3D", max_gap="1D",
                        min_neighbors=6, threshold=4.5, abs_floor=1.0,
                        time_basis="trading", session_timezone="America/New_York",
                        session_open="08:00", session_close="18:30")

# UI LOGIC: All algorithm settings can be changed through sliders and exact input boxes.
panel = show_filter(data, config=settings, cusip_col="CUSIP", time_col="time",
                    spread_col="spread", quantity_col="Quantity", timezone="UTC", unit="bp")
_ = panel.run()

## Use your own dataframe

Replace `data` above with your existing dataframe and supply its exact column names.
For example, BondCliQ data can map `CUSIP`, `time`, and `BM_SPREAD`.
If `BM_SPREAD` is in percentage points, use `unit="percentage points"`
and choose `abs_floor` in the same units (1 bp = 0.01 percentage points).
In Optional fitting rules, select your quantity column. Original amounts use
quantity_multiplier=1, so the default 1MM threshold is 1,000,000. Columns in
thousands or millions require multipliers of 1,000 or 1,000,000 respectively.
The distance rule is expressed in bp; explicitly set spread_units_per_bp=0.01
for percentage-point spreads (or 1 for spreads already in bp). Labels do not
convert observations. The default distance boundary is strictly greater than
10 bp, and quantities equal to 1MM are not treated as small trades.
Quantity alone does not reject a small trade. For uncertain statistical cases,
small and suspicious trades are rejected; otherwise valid known-size cases
are retained with status `retained_uncertain`. Strong algorithm outliers still
reject any size. Missing quantity leaves the algorithm decision unchanged.
The maximum-distance rule needs a bracketed local trend with sufficient support;
it does not construct evidence across a gap or infer a trend where none exists.
Enable **Require a verified cap check for fitting** (require_cap_support=True)
alongside the cap to exclude unverified rows from both hard and soft fitting.
Unverified support remains distinct from confirmed outliers. The plot and audits
distinguish retained trades within the cap from retained trades with no cap check.
Original algorithm evidence and the final fitting decision are both exported.
The engine preserves original rows and index; flags and reasons are added.

Three columns cannot identify whether a flagged trade was retail, distressed,
marked up/down, or charged a commission. Treat each flag as a review candidate.
Centered Hampel, local trend and reversal methods use future observations.
Causal robust EWMA uses only prior timestamps and marks unconfirmed jumps.
Offline Huber-TV jointly fits the whole gap-separated segment, while its local
residual scale excludes the target timestamp cohort. Solver failures abstain.
Two-sided local piecewise trends can retain a continuous uptrend/downtrend corner;
conflicting side predictions abstain as `ambiguous_transition` rather than
automatically declaring the observation clean or an execution outlier.
With `time_basis="trading"`, horizon/max_gap are cumulative configured open time.
The weekday calendar is a research assumption. Specify `holidays` or pass an
exact timezone-aware `session_schedule` to `filter_trades` for early closes.
Pass the same table to `show_filter(..., session_schedule=calendar)` to use it
in the notebook dashboard. Applied settings export its fingerprint and the
authoritative schedule CSV for reproducibility.

`panel.result` contains the applied review population: the selected bond by
default, or the full source after All bonds · batch Apply. The Export button saves
complete annotations for that population, bond statistics, settings and a chart.
Large charts report a bounded display sample; filtering uses the full population.
**Compare methods for this bond** uses the applied settings; flag counts alone
do not establish filtering accuracy.

In [ ]:
# Input: panel.result rows jf_row_id=[0,1,2],jf_status=['ok','outlier','ambiguous_transition'],
# jf_is_outlier=[False,True,False],jf_weight=[1,.15,0],jf_fit_eligible=[True,False,False].
# Output: hard_fit jf_row_id=[0],jf_fit_weight=[1]; soft_fit jf_row_id=[0,1],jf_fit_weight=[1,.15].
# Trick: Applied original spreads are retained; provisional/ambiguous/unsupported rows are excluded by default.
# CORE LOGIC: STEP 1 — Select explicit hard and soft fitting populations from the applied review.
hard_fit = select_fit_data(panel.result, policy="hard")
soft_fit = select_fit_data(panel.result, policy="soft")

# REPORTING LOGIC: Hard fit keeps jf_fit_eligible with weight 1; soft fit exposes jf_fit_weight.
# With optional rules off, unsupported/invalid/provisional/ambiguous/solver-failure rows are excluded.
# An enabled quantity rule can explicitly retain valid uncertain rows; their uncertainty stays auditable.
# Soft weights limit influence; they are neither probabilities nor inverse variances.
print(f"Source: {len(data):,}; applied review: {len(panel.result):,}; hard-fit candidates: {len(hard_fit):,}; soft-fit candidates: {len(soft_fit):,}")